## Step 1: Build the Initial Private Knowledge Base

In [3]:
import requests
import pandas as pd
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, XSD
import math

SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"

EX = Namespace("http://example.org/")

# We choose different big football clubs to have more entities
CLUBS = {
    "Manchester United": "Q18656",
    "FC Barcelona":      "Q7156",
    "Real Madrid":       "Q8682",
    "Liverpool":         "Q1130849",
    "Bayern Munich":     "Q15789",
    "Paris Saint-Germain": "Q583",
    "Juventus":            "Q17050",
    "Manchester City":     "Q50602",
}

def fetch_players_for_club(club_qid: str, club_name: str) -> list:
    query = f"""
    PREFIX wd: <http://www.wikidata.org/entity/>
    PREFIX wdt: <http://www.wikidata.org/prop/direct/>

    SELECT ?player ?playerLabel
           ?pos ?posLabel
           ?dob
           ?country ?countryLabel
           ?height
           ?birthPlace ?birthPlaceLabel
    WHERE {{
      ?player wdt:P31 wd:Q5 ;
              wdt:P106 wd:Q937857 ;
              wdt:P54 wd:{club_qid} .

      OPTIONAL {{ ?player wdt:P413 ?pos . }}
      OPTIONAL {{ ?player wdt:P569 ?dob . }}
      OPTIONAL {{ ?player wdt:P27 ?country . }}
      OPTIONAL {{ ?player wdt:P2048 ?height . }}
      OPTIONAL {{ ?player wdt:P19 ?birthPlace . }}

      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en" . }}
    }}
    """
    resp = requests.get(
        SPARQL_ENDPOINT,
        params={"query": query, "format": "json"},
        headers={"User-Agent": "FootballKB/1.0"}
    )
    results = resp.json()["results"]["bindings"]
    print(f"  {club_name}: {len(results)} joueurs")
    return results

# Collect for all clubs
all_results = []
for club_name, club_qid in CLUBS.items():
    rows = fetch_players_for_club(club_qid, club_name)
    # We add the club if origin in each row
    for r in rows:
        r["_club_qid"]  = club_qid
        r["_club_name"] = club_name
    all_results.extend(rows)

print(f"\nTotal lines collected : {len(all_results)}")

  Manchester United: 1740 joueurs
  FC Barcelona: 1524 joueurs
  Real Madrid: 925 joueurs
  Liverpool: 1547 joueurs
  Bayern Munich: 901 joueurs
  Paris Saint-Germain: 0 joueurs
  Juventus: 0 joueurs
  Manchester City: 1511 joueurs

Total lines collected : 8148


In [4]:
def qid_from_wd_uri(uri):
    if not uri or not isinstance(uri, str):
        return None
    if "/entity/" in uri:
        return uri.rsplit("/entity/", 1)[1]
    return None

def to_xsd_date(value):
    if value is None or not isinstance(value, str):
        return None
    if "T" in value:
        value = value.split("T", 1)[0]
    if len(value) == 10 and value[4] == "-" and value[7] == "-":
        y, m, d = value[0:4], value[5:7], value[8:10]
        if y.isdigit() and m.isdigit() and d.isdigit():
            if 1 <= int(m) <= 12 and 1 <= int(d) <= 31:
                return value
    return None

g = Graph()
g.bind("ex",   EX)
g.bind("wd",   Namespace("http://www.wikidata.org/entity/"))
g.bind("wdt",  Namespace("http://www.wikidata.org/prop/direct/"))
g.bind("rdfs", RDFS)
g.bind("xsd",  XSD)

seen_players = set()

for r in all_results:
    # Player
    raw_player = r["player"]["value"]
    qid = qid_from_wd_uri(raw_player)
    if not qid or qid in seen_players:
        continue
    seen_players.add(qid)

    player_uri = EX[f"player/{qid}"]
    g.add((player_uri, RDF.type,    EX.FootballPlayer))
    g.add((player_uri, RDFS.label,  Literal(r.get("playerLabel", {}).get("value", qid), lang="en")))

    # Club
    club_qid  = r["_club_qid"]
    club_name = r["_club_name"]
    club_uri  = EX[f"club/{club_qid}"]
    g.add((club_uri,   RDF.type,   EX.Club))
    g.add((club_uri,   RDFS.label, Literal(club_name, lang="en")))
    g.add((player_uri, EX.playsFor, club_uri))

    # Position
    if "pos" in r:
        pos_qid = qid_from_wd_uri(r["pos"]["value"])
        if pos_qid:
            pos_uri = EX[f"position/{pos_qid}"]
            g.add((pos_uri,    RDF.type,   EX.Position))
            g.add((pos_uri,    RDFS.label, Literal(r.get("posLabel", {}).get("value", pos_qid), lang="en")))
            g.add((player_uri, EX.hasPosition, pos_uri))

    # Nationality
    if "country" in r:
        c_qid = qid_from_wd_uri(r["country"]["value"])
        if c_qid:
            c_uri = EX[f"country/{c_qid}"]
            g.add((c_uri,      RDF.type,   EX.Country))
            g.add((c_uri,      RDFS.label, Literal(r.get("countryLabel", {}).get("value", c_qid), lang="en")))
            g.add((player_uri, EX.hasNationality, c_uri))

    # Date of birth
    dob = to_xsd_date(r.get("dob", {}).get("value"))
    if dob:
        g.add((player_uri, EX.birthDate, Literal(dob, datatype=XSD.date)))

    # Height
    if "height" in r:
        try:
            h = float(r["height"]["value"])
            if not math.isnan(h) and 1.0 < h < 2.5:
                g.add((player_uri, EX.heightM, Literal(round(h, 2), datatype=XSD.decimal)))
        except:
            pass

    # Birth place
    if "birthPlace" in r:
        bp_qid = qid_from_wd_uri(r["birthPlace"]["value"])
        if bp_qid:
            bp_uri = EX[f"place/{bp_qid}"]
            g.add((bp_uri,     RDF.type,   EX.Place))
            g.add((bp_uri,     RDFS.label, Literal(r.get("birthPlaceLabel", {}).get("value", bp_qid), lang="en")))
            g.add((player_uri, EX.birthPlace, bp_uri))

g.serialize(destination="private_kb.ttl", format="turtle")
print("Saved: private_kb.ttl")
print("Triples:", len(g))

Saved: private_kb.ttl
Triples: 46085


In [5]:
# Stats
subjects = set(g.subjects())
objects  = set(o for o in g.objects() if isinstance(o, URIRef))
entities = subjects | objects
print("Entities   :", len(entities))
print("Relations :", len(set(g.predicates())))

types = {}
for s,p,o in g:
    if str(p).endswith("type"):
        t = str(o).split("/")[-1]
        types[t] = types.get(t, 0) + 1
for k,v in sorted(types.items(), key=lambda x:-x[1]):
    print(f"  {k:<20} : {v}")

Entities   : 8424
Relations : 8
  FootballPlayer       : 5975
  Place                : 2306
  Country              : 102
  Position             : 30
  Club                 : 6


## Step 2: Entity Linking with Open Knowledge Bases

In [6]:
from rdflib.namespace import OWL

WD = Namespace("http://www.wikidata.org/entity/")
g.bind("wd",  WD)
g.bind("owl", OWL)

def as_wd_entity(ex_uri: URIRef):
    """
    Convert ex:player/Q123 → wd:Q123
    Work for player/, club/, position/, country/, place/
    """
    s = str(ex_uri)
    # We try to find the QID in the last segment
    last = s.rsplit("/", 1)[-1]
    if last.startswith("Q") and last[1:].isdigit():
        return WD[last]
    return None

linked_count = 0
all_uris = set(g.subjects()) | set(o for o in g.objects() if isinstance(o, URIRef))

for uri in all_uris:
    wd_uri = as_wd_entity(uri)
    if wd_uri is not None:
        g.add((uri, OWL.sameAs, wd_uri))
        linked_count += 1

print(f"owl:sameAs ajoutés : {linked_count}")
print(f"Triples total      : {len(g)}")

owl:sameAs ajoutés : 8419
Triples total      : 54504


In [7]:
# Vérification :  show 10 examples
count = 0
for s, p, o in g.triples((None, OWL.sameAs, None)):
    print(s, "owl:sameAs", o)
    count += 1
    if count >= 10:
        break

http://example.org/player/Q22342683 owl:sameAs http://www.wikidata.org/entity/Q22342683
http://example.org/player/Q6134778 owl:sameAs http://www.wikidata.org/entity/Q6134778
http://example.org/player/Q877144 owl:sameAs http://www.wikidata.org/entity/Q877144
http://example.org/place/Q10305 owl:sameAs http://www.wikidata.org/entity/Q10305
http://example.org/player/Q3296570 owl:sameAs http://www.wikidata.org/entity/Q3296570
http://example.org/place/Q3838 owl:sameAs http://www.wikidata.org/entity/Q3838
http://example.org/place/Q1397011 owl:sameAs http://www.wikidata.org/entity/Q1397011
http://example.org/player/Q7819419 owl:sameAs http://www.wikidata.org/entity/Q7819419
http://example.org/player/Q20004514 owl:sameAs http://www.wikidata.org/entity/Q20004514
http://example.org/player/Q755567 owl:sameAs http://www.wikidata.org/entity/Q755567


In [8]:
g.serialize(destination="private_kb_linked.ttl", format="turtle")
print("Saved: private_kb_linked.ttl")
print("Triples:", len(g))

Saved: private_kb_linked.ttl
Triples: 54504


In [9]:
# Table for mapping
import pandas as pd

rows = []
for s, p, o in g.triples((None, OWL.sameAs, None)):
    rows.append({
        "Private Entity":  str(s),
        "External URI":    str(o),
        "Confidence":      1.0
    })

df_mapping = pd.DataFrame(rows)
df_mapping.to_csv("mapping_table.csv", index=False)
print(df_mapping.head(10))
print(f"\nTotal entities linked : {len(df_mapping)}")

                        Private Entity  \
0  http://example.org/player/Q22342683   
1   http://example.org/player/Q6134778   
2    http://example.org/player/Q877144   
3      http://example.org/place/Q10305   
4   http://example.org/player/Q3296570   
5       http://example.org/place/Q3838   
6    http://example.org/place/Q1397011   
7   http://example.org/player/Q7819419   
8  http://example.org/player/Q20004514   
9    http://example.org/player/Q755567   

                               External URI  Confidence  
0  http://www.wikidata.org/entity/Q22342683         1.0  
1   http://www.wikidata.org/entity/Q6134778         1.0  
2    http://www.wikidata.org/entity/Q877144         1.0  
3     http://www.wikidata.org/entity/Q10305         1.0  
4   http://www.wikidata.org/entity/Q3296570         1.0  
5      http://www.wikidata.org/entity/Q3838         1.0  
6   http://www.wikidata.org/entity/Q1397011         1.0  
7   http://www.wikidata.org/entity/Q7819419         1.0  
8  http://www.w

## Step 3: Predicate Alignment Using SPARQL

In [10]:
from rdflib.namespace import OWL

WDT = Namespace("http://www.wikidata.org/prop/direct/")
g.bind("wdt", WDT)

# Alignment: private predicate -> equivalent Wikidata property
predicate_alignment = {
    EX.playsFor:       WDT.P54,    # member of sports team
    EX.hasPosition:    WDT.P413,   # position played
    EX.hasNationality: WDT.P27,    # country of citizenship
    EX.birthPlace:     WDT.P19,    # place of birth
    EX.birthDate:      WDT.P569,   # date of birth
    EX.heightM:        WDT.P2048,  # height
}

added = 0
for p_ex, p_wdt in predicate_alignment.items():
    g.add((p_ex, OWL.equivalentProperty, p_wdt))
    added += 1

print(f"Alignements added : {added}")
print(f"Triples total       : {len(g)}")

Alignements added : 6
Triples total       : 54510


In [11]:
# Verification
for p_ex, p_wdt in predicate_alignment.items():
    ok = (p_ex, OWL.equivalentProperty, p_wdt) in g
    print(f"{str(p_ex).split('/')[-1]:<20} ≡  {str(p_wdt).split('/')[-1]}  ->  {'OK' if ok else 'MISSING'}")

playsFor             ≡  P54  ->  OK
hasPosition          ≡  P413  ->  OK
hasNationality       ≡  P27  ->  OK
birthPlace           ≡  P19  ->  OK
birthDate            ≡  P569  ->  OK
heightM              ≡  P2048  ->  OK


In [12]:
g.serialize(destination="private_kb_linked_aligned.ttl", format="turtle")
print("Saved: private_kb_linked_aligned.ttl")
print("Triples:", len(g))

Saved: private_kb_linked_aligned.ttl
Triples: 54510


## Step 4: KB Expansion via SPARQL

In [13]:
import requests, time
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL, XSD

SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"
WD  = Namespace("http://www.wikidata.org/entity/")
WDT = Namespace("http://www.wikidata.org/prop/direct/")

# Load the aligned KB
g_exp = Graph()
g_exp.parse("private_kb_linked_aligned.ttl", format="turtle")
print(f"Starting KB : {len(g_exp)} triples")

# Extract all anchor QIDs from owl:sameAs links
anchor_qids = set()
for s, p, o in g_exp.triples((None, OWL.sameAs, None)):
    qid = str(o).split("/")[-1]
    if qid.startswith("Q") and qid[1:].isdigit():
        anchor_qids.add(qid)

print(f"Anchor entities found : {len(anchor_qids)}")

Starting KB : 54510 triples
Anchor entities found : 8404


In [19]:
# Allowed Wikidata predicates (avoids heavy/useless literals)
ALLOWED_PIDS = [
    "P54",   # member of sports team
    "P27",   # country of citizenship
    "P413",  # position played
    "P569",  # date of birth
    "P19",   # place of birth
    "P20",   # place of death
    "P106",  # occupation
    "P166",  # award received
    "P17",   # country
    "P131",  # located in administrative entity
    "P31",   # instance of
    "P641",  # sport
    "P118",  # league
    "P115",  # home venue / stadium
    "P571",  # inception (founding date)
    "P286",  # head coach
    "P1532", # country for sport
    "P2048", # height
    "P1351",   # number of goals scored
    "P6509",   # total goals in career
    "P2276",   # UEFA club competition appearances
    "P552",    # handedness (left/right foot)
    "P741",    # playing hand
    "P1532",   # country for sport
    "P97",     # noble title (for captaincy)
    "P69",     # educated at
    "P101",    # field of work
    "P1344",   # participant in (World Cup, Euro, etc.)
    "P555",    # shirt number
    "P6087",   # coach of sports team
    "P2522",   # victory
    "P1346",   # winner of
    "P1923",   # participating team
    "P585",    # point in time
    "P580",    # start time
    "P582",    # end time
]
FILTER_PIDS = " || ".join([f"?p = wdt:{pid}" for pid in ALLOWED_PIDS])

def add_triple(g, s_uri, p_uri, row_o):
    """Add a triple to the graph from a SPARQL result row"""
    o_val  = row_o.get("value", "")
    o_type = row_o.get("type", "")
    if not o_val:
        return
    if o_type == "uri":
        g.add((s_uri, p_uri, URIRef(o_val)))
    elif o_type == "literal":
        lang  = row_o.get("xml:lang", "")
        dtype = row_o.get("datatype", "")
        if lang:
            g.add((s_uri, p_uri, Literal(o_val, lang=lang)))
        elif dtype:
            g.add((s_uri, p_uri, Literal(o_val, datatype=URIRef(dtype))))
        else:
            g.add((s_uri, p_uri, Literal(o_val)))

def batch_one_hop(qids: list, limit=500) -> list:
    """1-hop expansion on a batch of QIDs in a single SPARQL query"""
    values = " ".join([f"wd:{q}" for q in qids])
    query = f"""
    PREFIX wd:  <http://www.wikidata.org/entity/>
    PREFIX wdt: <http://www.wikidata.org/prop/direct/>

    SELECT ?s ?p ?o WHERE {{
      VALUES ?s {{ {values} }}
      ?s ?p ?o .
      FILTER({FILTER_PIDS})
      FILTER(!isLiteral(?o) || LANG(?o) = "en" || LANG(?o) = "")
    }}
    LIMIT {limit}
    """
    try:
        resp = requests.get(
            SPARQL_ENDPOINT,
            params={"query": query, "format": "json"},
            headers={"User-Agent": "FootballKB/1.0"},
            timeout=30
        )
        if resp.status_code == 200:
            return resp.json()["results"]["bindings"]
        return []
    except:
        return []

def two_hop_batch(qids: list, limit=300) -> list:
    """2-hop expansion: player -> club -> club info. Fetches properties of clubs linked to a batch of players"""
    values = " ".join([f"wd:{q}" for q in qids])
    query = f"""
    PREFIX wd:  <http://www.wikidata.org/entity/>
    PREFIX wdt: <http://www.wikidata.org/prop/direct/>

    SELECT ?club ?p ?o WHERE {{
      VALUES ?player {{ {values} }}
      ?player wdt:P54 ?club .
      ?club ?p ?o .
      FILTER({FILTER_PIDS})
      FILTER(!isLiteral(?o) || LANG(?o) = "en" || LANG(?o) = "")
    }}
    LIMIT {limit}
    """
    try:
        resp = requests.get(
            SPARQL_ENDPOINT,
            params={"query": query, "format": "json"},
            headers={"User-Agent": "FootballKB/1.0"},
            timeout=30
        )
        if resp.status_code == 200:
            return resp.json()["results"]["bindings"]
        return []
    except:
        return []

def predicate_expansion(pid: str, limit=5000) -> list:
    """ Predicate-controlled expansion:fetch all footballers from Wikidata that have a given property"""
    query = f"""
    PREFIX wd:  <http://www.wikidata.org/entity/>
    PREFIX wdt: <http://www.wikidata.org/prop/direct/>

    SELECT ?s ?o WHERE {{
      ?s wdt:P31  wd:Q5 .
      ?s wdt:P106 wd:Q937857 .
      ?s wdt:{pid} ?o .
    }}
    LIMIT {limit}
    """
    try:
        resp = requests.get(
            SPARQL_ENDPOINT,
            params={"query": query, "format": "json"},
            headers={"User-Agent": "FootballKB/1.0"},
            timeout=30
        )
        if resp.status_code == 200:
            return resp.json()["results"]["bindings"]
        return []
    except:
        return []

In [20]:
# Phase 1: 1-hop expansion in batches of 50 QIDs
# We only expand players (not places/countries) to keep it manageable
print("=== Phase 1: 1-hop batch expansion (players only) ===")

# Keep only player QIDs for 1-hop (reduces from 7000 to 5000)
player_qids = [
    qid for qid in anchor_qids
    if any(
        str(s).endswith(qid)
        for s in g_exp.subjects(RDF.type, URIRef("http://example.org/FootballPlayer"))
    )
]
print(f"Player QIDs to expand : {len(player_qids)}")

# Process in batches of 50
BATCH_SIZE = 50
batches = [player_qids[i:i+BATCH_SIZE] for i in range(0, len(player_qids), BATCH_SIZE)]
print(f"Number of batches     : {len(batches)}")

for i, batch in enumerate(batches):
    rows = batch_one_hop(batch, limit=500)
    for row in rows:
        s_uri = URIRef(row["s"]["value"])
        p_uri = URIRef(row["p"]["value"])
        add_triple(g_exp, s_uri, p_uri, row["o"])

    if (i+1) % 10 == 0 or (i+1) == len(batches):
        print(f"  Batch [{i+1}/{len(batches)}] — triples : {len(g_exp):,}")

    time.sleep(0.8)  # Respect Wikidata rate limit

print(f"\nAfter phase 1 : {len(g_exp):,} triples")

=== Phase 1: 1-hop batch expansion (players only) ===
Player QIDs to expand : 5975
Number of batches     : 120
  Batch [10/120] — triples : 149,832
  Batch [20/120] — triples : 150,239
  Batch [30/120] — triples : 150,763
  Batch [40/120] — triples : 151,148
  Batch [50/120] — triples : 151,645
  Batch [60/120] — triples : 152,027
  Batch [70/120] — triples : 152,460
  Batch [80/120] — triples : 152,888
  Batch [90/120] — triples : 153,323
  Batch [100/120] — triples : 153,729
  Batch [110/120] — triples : 154,154
  Batch [120/120] — triples : 154,496

After phase 1 : 154,496 triples


In [21]:
# Phase 2: 2-hop expansion (player -> club -> club info)
print("=== Phase 2: 2-hop expansion (player -> club) ===")

# Use only first 200 players for 2-hop to save time
sample_players = player_qids[:200]
batches_2hop   = [sample_players[i:i+BATCH_SIZE] for i in range(0, len(sample_players), BATCH_SIZE)]

for i, batch in enumerate(batches_2hop):
    rows = two_hop_batch(batch, limit=300)
    for row in rows:
        club_uri = URIRef(row["club"]["value"])
        p_uri    = URIRef(row["p"]["value"])
        add_triple(g_exp, club_uri, p_uri, row["o"])

    if (i+1) % 5 == 0 or (i+1) == len(batches_2hop):
        print(f"  Batch [{i+1}/{len(batches_2hop)}] — triples : {len(g_exp):,}")

    time.sleep(0.8)

print(f"\nAfter phase 2 : {len(g_exp):,} triples")

=== Phase 2: 2-hop expansion (player -> club) ===
  Batch [4/4] — triples : 154,851

After phase 2 : 154,851 triples


In [22]:
# Phase 3: Predicate-controlled expansion
# Fetches all footballers on Wikidata that have these properties
print("=== Phase 3: Predicate-controlled expansion ===")

controlled_preds = [
    ("P54",   3000),  # club membership
    ("P27",   3000),  # nationality
    ("P166",  2000),  # awards received
    ("P413",  2000),  # position played
    ("P19",   2000),  # place of birth
    ("P569",  2000),  # date of birth
    ("P106",  2000),  # occupation
    ("P20",   2000),  # place of death
    ("P17",   2000),  # country
    ("P131",  2000),  # located in administrative entity
    ("P31",   2000),  # instance of
    ("P641",  2000),  # sport
    ("P118",  2000),  # league
    ("P115",  2000),  # home venue
    ("P571",  2000),  # founding date
    ("P286",  2000),  # head coach
    ("P1532", 2000),  # country for sport
    ("P2048", 2000),  # height
    ("P21",   2000),  # sex or gender
    ("P40",   1000),  # child
    ("P22",   1000),  # father
    ("P25",   1000),  # mother
    ("P26",   1000),  # spouse
    ("P108",  2000),  # employer
    ("P award",1000), # notable work
    ("P1344", 2000),  # participant in
    ("P medal",1000), # participant in
    ("P460",  1000),  # said to be the same as
    ("P641",  1000),  # sport
    ("P2046", 1000),  # area
    ("P276",  1000),  # location
    ("P159",  1000),  # headquarters
    ("P856",  1000),  # official website
    ("P18",   1000),  # image
    ("P41",   1000),  # flag image
    ("P94",   1000),  # coat of arms
    ("P242",  1000),  # locator map
    ("P281",  1000),  # postal code
    ("P625",  1000),  # coordinate location
    ("P30",   1000),  # continent
    ("P36",   1000),  # capital
    ("P37",   1000),  # official language
    ("P38",   1000),  # currency
    ("P47",   1000),  # shares border with
    ("P150",  1000),  # contains administrative territorial entity
    ("P190",  1000),  # twinned administrative body
    ("P194",  1000),  # legislative body
    ("P6",    1000),  # head of government
    ("P35",   1000),  # head of state
    ("P530",  1000),  # diplomatic relation
    ("P1566", 1000),  # GeoNames ID
    ("P910",  1000),  # topic's main category
]

for pid, limit in controlled_preds:
    rows  = predicate_expansion(pid, limit=limit)
    p_uri = URIRef(f"http://www.wikidata.org/prop/direct/{pid}")
    for row in rows:
        s_uri = URIRef(row["s"]["value"])
        add_triple(g_exp, s_uri, p_uri, row["o"])
    print(f"  wdt:{pid} -> +{len(rows):,} rows  (total triples: {len(g_exp):,})")
    time.sleep(1.5)

print(f"\nAfter phase 3 : {len(g_exp):,} triples")

=== Phase 3: Predicate-controlled expansion ===
  wdt:P54 -> +3,000 rows  (total triples: 155,189)
  wdt:P27 -> +3,000 rows  (total triples: 155,990)
  wdt:P166 -> +2,000 rows  (total triples: 156,028)
  wdt:P413 -> +2,000 rows  (total triples: 156,308)
  wdt:P19 -> +2,000 rows  (total triples: 156,582)
  wdt:P569 -> +2,000 rows  (total triples: 156,990)
  wdt:P106 -> +2,000 rows  (total triples: 157,293)
  wdt:P20 -> +2,000 rows  (total triples: 157,346)
  wdt:P17 -> +4 rows  (total triples: 157,346)
  wdt:P131 -> +1 rows  (total triples: 157,346)
  wdt:P31 -> +2,000 rows  (total triples: 157,589)
  wdt:P641 -> +2,000 rows  (total triples: 157,817)
  wdt:P118 -> +2,000 rows  (total triples: 158,246)
  wdt:P115 -> +0 rows  (total triples: 158,246)
  wdt:P571 -> +1 rows  (total triples: 158,246)
  wdt:P286 -> +59 rows  (total triples: 158,246)
  wdt:P1532 -> +2,000 rows  (total triples: 158,612)
  wdt:P2048 -> +2,000 rows  (total triples: 158,994)
  wdt:P21 -> +2,000 rows  (total triple

  wdt:P award -> +0 rows  (total triples: 159,431)
  wdt:P1344 -> +2,000 rows  (total triples: 159,725)


  wdt:P medal -> +0 rows  (total triples: 159,725)
  wdt:P460 -> +40 rows  (total triples: 159,725)
  wdt:P641 -> +1,000 rows  (total triples: 159,821)
  wdt:P2046 -> +0 rows  (total triples: 159,821)
  wdt:P276 -> +0 rows  (total triples: 159,821)
  wdt:P159 -> +1 rows  (total triples: 159,821)
  wdt:P856 -> +1,000 rows  (total triples: 159,821)
  wdt:P18 -> +1,000 rows  (total triples: 159,947)
  wdt:P41 -> +1 rows  (total triples: 159,947)
  wdt:P94 -> +8 rows  (total triples: 159,947)
  wdt:P242 -> +0 rows  (total triples: 159,947)
  wdt:P281 -> +0 rows  (total triples: 159,947)
  wdt:P625 -> +0 rows  (total triples: 159,947)
  wdt:P30 -> +0 rows  (total triples: 159,947)
  wdt:P36 -> +0 rows  (total triples: 159,947)
  wdt:P37 -> +1 rows  (total triples: 159,947)
  wdt:P38 -> +0 rows  (total triples: 159,947)
  wdt:P47 -> +0 rows  (total triples: 159,947)
  wdt:P150 -> +0 rows  (total triples: 159,947)
  wdt:P190 -> +0 rows  (total triples: 159,947)
  wdt:P194 -> +0 rows  (total t

In [23]:
# Final stats and save
entities  = set(str(s) for s,p,o in g_exp if isinstance(s, URIRef))
entities |= set(str(o) for s,p,o in g_exp if isinstance(o, URIRef))
relations = set(str(p) for s,p,o in g_exp)

print(f"\n{'─'*40}")
print(f"  FINAL STATISTICS")
print(f"{'─'*40}")
print(f"  Triples   : {len(g_exp):>10,}")
print(f"  Entities  : {len(entities):>10,}")
print(f"  Relations : {len(relations):>10,}")
print(f"\nThreshold check (lab requirements):")
print(f"  50k-200k triples  : {'OK' if 50000  <= len(g_exp)     <= 200000 else 'OUT OF RANGE'} ({len(g_exp):,})")
print(f"  5k-30k entities   : {'OK' if 5000   <= len(entities)  <= 30000  else 'OUT OF RANGE'} ({len(entities):,})")
print(f"  50-200 relations  : {'OK' if 50     <= len(relations) <= 200    else 'OUT OF RANGE'} ({len(relations):,})")

# Save in both formats
g_exp.serialize("expanded_kb.ttl", format="turtle")
g_exp.serialize("expanded_kb.nt",  format="nt")
print(f"\nSaved: expanded_kb.ttl")
print(f"Saved: expanded_kb.nt  (for KGE)")


────────────────────────────────────────
  FINAL STATISTICS
────────────────────────────────────────
  Triples   :    159,947
  Entities  :     40,555
  Relations :         51

Threshold check (lab requirements):
  50k-200k triples  : OK (159,947)
  5k-30k entities   : OUT OF RANGE (40,555)
  50-200 relations  : OK (51)

Saved: expanded_kb.ttl
Saved: expanded_kb.nt  (for KGE)


In [24]:
from google.colab import files

files.download('private_kb.ttl')
files.download('private_kb_linked.ttl')
files.download('private_kb_linked_aligned.ttl')
files.download('expanded_kb.ttl')
files.download('expanded_kb.nt')
files.download('mapping_table.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>